In [60]:
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextBox, LTChar
from collections import defaultdict
import re

In [61]:
en_short_paper = 'resnet.pdf'
cn_short_paper = 'Paper.pdf'
path = 'G:\\Postgraduation\\研一下\\project-论文摘要推文生成\\测试paper\\测试paper\\'

In [62]:
#按页提取文本、每页字数和每个字符的字体名和字符大小
def extract_text_and_fonts_by_page(pdf_path):
    text_by_page = {}
    word_count_by_page = {}
    fonts_by_page = {}

    for page_number, page_layout in enumerate(extract_pages(pdf_path)):
        text_elements = []
        font_details = []

        for element in page_layout:
            if isinstance(element, LTTextBox):
                text_block = ""
                for text_line in element:
                    text_block += text_line.get_text()
                    
                    if hasattr(text_line, '__iter__'):  # 确保text_line是可迭代的
                        for character in text_line:
                            if isinstance(character, LTChar): 
                                font_details.append((character.get_text(), character.fontname, character.size))
                    else:
                    # 处理非可迭代的text_line情况
                        if isinstance(text_line, LTChar):
                            font_details.append((text_line.get_text(), text_line.fontname, text_line.size))
                     
                text_elements.append(text_block)
        
        page_text = "\n".join(text_elements)
        text_by_page[page_number + 1] = page_text
        word_count_by_page[page_number + 1] = len(page_text)  
        fonts_by_page[page_number + 1] = font_details

    return text_by_page, word_count_by_page, fonts_by_page



In [63]:
texts, counts, fonts = extract_text_and_fonts_by_page(path + en_short_paper)

In [64]:
full_pdf_text = ''.join(f'{key}{value}' for key, value in texts.items())

In [65]:
#合并fonts，把value抽取出来作为一个列表
combined_fonts = []
for value in fonts.values():
    combined_fonts.extend(value)

结构化提取

In [66]:
def find_main_and_titles_fonts(fonts):
    font_counts = defaultdict(int)
    for page_fonts in fonts.values():
        for _, font_name, font_size in page_fonts:
            font_counts[(font_name, font_size)] += 1
    
    # 找出正文文本的字体样式
    main_text_style = max(font_counts, key=font_counts.get)
    main_text_size = main_text_style[1]

    # 收集所有比正文字体大的可能的标题字体样式
    titles_fonts = [(font_name, font_size) for (font_name, font_size), _ in font_counts.items() if font_size > main_text_size]

    return main_text_style, titles_fonts

In [67]:
main_text_style, titles_fonts = find_main_and_titles_fonts(fonts)
main_text_size = main_text_style[1] #正文字体大小

In [68]:
main_text_size

9.962600000000009

In [69]:
def extract_possible_titles_by_re(text):
    matches_titles = {}
    for page, text in texts.items():
        pattern = r'([^\n]+)\n\n'
        matches = re.findall(pattern, text)
        
        # 将处理后的文本存储在新的字典中
        matches_titles[page] = matches
    return matches_titles

In [70]:
# 调用函数并打印结果
result = extract_possible_titles_by_re(texts)
print(result)

{1: ['Deep Residual Learning for Image Recognition', 'Kaiming He', 'Xiangyu Zhang', 'Shaoqing Ren', 'Jian Sun', 'Microsoft Research', '{', '1', ']', '[', 'a', 'Abstract', 'on CIFAR-10 with 100 and 1000 layers.', 'ization, COCO detection, and COCO segmentation.', '1. Introduction', 'trivial visual recognition tasks [8, 12, 7, 32, 27] have also', '1http://image-net.org/challenges/LSVRC/2015/', 'http://mscoco.org/dataset/#detections-challenge2015.', 'on ImageNet is presented in Fig. 4.', 'greatly beneﬁted from very deep models.', 'propagation [22].', 'our experiments. Fig. 1 shows a typical example.', 'our current solvers on hand are unable to ﬁnd solutions that'], 2: ['Figure 2. Residual learning: a building block.', 'it is applicable in other vision and non-vision problems.', '(or unable to do so in feasible time).', '2. Related Work', 'learning framework.', 'of nonlinear layers.', '(x) :=', 'H', 'H', '−', 'F', 'F', 'The formulation of', 'without modifying the solvers.', 'sults substant

In [71]:
title_min_len = 5
title_max_len = 50

def clean_results(results):
    new_results = {}
    for page, strings in results.items():
        new_results[page] = [
            s for s in strings
            if title_min_len <= len(s) <= title_max_len
            and not re.search(r"[\t\n\r\f\v\u200B]", s)
            and not re.search(r"(?i)(table|figure)", s)
        ]
    return new_results

In [72]:
# 示例使用
filtered_result = clean_results(result)
print(filtered_result)

{1: ['Deep Residual Learning for Image Recognition', 'Kaiming He', 'Xiangyu Zhang', 'Shaoqing Ren', 'Jian Sun', 'Microsoft Research', 'Abstract', 'on CIFAR-10 with 100 and 1000 layers.', 'ization, COCO detection, and COCO segmentation.', '1. Introduction', '1http://image-net.org/challenges/LSVRC/2015/', 'on ImageNet is presented in Fig. 4.', 'greatly beneﬁted from very deep models.', 'propagation [22].', 'our experiments. Fig. 1 shows a typical example.'], 2: ['(or unable to do so in feasible time).', '2. Related Work', 'learning framework.', 'of nonlinear layers.', '(x) :=', 'The formulation of', 'without modifying the solvers.', 'sults substantially better than previous networks.', 'tive than encoding original vectors.', 'or preconditioning can simplify the optimization.'], 3: ['extremely increased depth (e.g., over 100 layers).', '3. Deep Residual Learning', '3.1. Residual Learning', 'Let us consider', 'the ease of learning might be different.', 'ers toward zero to approach identity

In [73]:
#解决fonts的索引匹配问题,先处理texts
def clean_texts(texts):
    new_texts = {}
    for page, text in texts.items():
        # 去除文本里所有的空格
        trimmed_text = text.replace(" ", "")
        # 使用正则表达式移除特殊字符
        cleaned_text = re.sub(r"[\t\n\r\f\v\u200B]+", "", trimmed_text)
        # 将处理后的文本存储在新的字典中
        new_texts[page] = cleaned_text
    return new_texts


In [74]:
new_texts = clean_texts(texts)  #索引和fonts一一对应
print(new_texts)

{1: 'DeepResidualLearningforImageRecognitionKaimingHeXiangyuZhangShaoqingRenJianSunMicrosoftResearch@microsoft.comkahe,v-xiangz,v-shren,jiansun}{5102ceD01]VC.sc[1v58330.2151:viXraAbstractDeeperneuralnetworksaremoredifﬁculttotrain.Wepresentaresiduallearningframeworktoeasethetrainingofnetworksthataresubstantiallydeeperthanthoseusedpreviously.Weexplicitlyreformulatethelayersaslearn-ingresidualfunctionswithreferencetothelayerinputs,in-steadoflearningunreferencedfunctions.Weprovidecom-prehensiveempiricalevidenceshowingthattheseresidualnetworksareeasiertooptimize,andcangainaccuracyfromconsiderablyincreaseddepth.OntheImageNetdatasetweevaluateresidualnetswithadepthofupto152layers—8×deeperthanVGGnets[41]butstillhavinglowercomplex-ity.Anensembleoftheseresidualnetsachieves3.57%errorontheImageNettestset.Thisresultwonthe1stplaceontheILSVRC2015classiﬁcationtask.WealsopresentanalysisonCIFAR-10with100and1000layers.Thedepthofrepresentationsisofcentralimportanceformanyvisualrecognitiontasks.Solelyduetoo

In [75]:
#处理标题，把可能是标题的字符串中的空格删去，以便与new_texts匹配
def remove_spaces_from_strings(filtered_result):
    result_for_match = {}
    for page, strings in filtered_result.items():
        # 对每个字符串移除空格
        no_space_strings = [s.replace(" ", "") for s in strings]
        result_for_match[page] = no_space_strings
    return result_for_match

In [76]:
# 处理数据
result_for_match = remove_spaces_from_strings(filtered_result)
print(result_for_match)

{1: ['DeepResidualLearningforImageRecognition', 'KaimingHe', 'XiangyuZhang', 'ShaoqingRen', 'JianSun', 'MicrosoftResearch', 'Abstract', 'onCIFAR-10with100and1000layers.', 'ization,COCOdetection,andCOCOsegmentation.', '1.Introduction', '1http://image-net.org/challenges/LSVRC/2015/', 'onImageNetispresentedinFig.4.', 'greatlybeneﬁtedfromverydeepmodels.', 'propagation[22].', 'ourexperiments.Fig.1showsatypicalexample.'], 2: ['(orunabletodosoinfeasibletime).', '2.RelatedWork', 'learningframework.', 'ofnonlinearlayers.', '(x):=', 'Theformulationof', 'withoutmodifyingthesolvers.', 'sultssubstantiallybetterthanpreviousnetworks.', 'tivethanencodingoriginalvectors.', 'orpreconditioningcansimplifytheoptimization.'], 3: ['extremelyincreaseddepth(e.g.,over100layers).', '3.DeepResidualLearning', '3.1.ResidualLearning', 'Letusconsider', 'theeaseoflearningmightbedifferent.', 'erstowardzerotoapproachidentitymappings.', 'pingsprovidereasonablepreconditioning.', '3.2.IdentityMappingbyShortcuts', 'weconsid

In [77]:
#匹配
def annotate_titles_with_font_info(filtered_result, new_texts, fonts):
    annotated_results = {}
    for page, titles in filtered_result.items():
        annotated_titles = []
        for title in titles:
            start_index = new_texts[page].find(title)
            if start_index != -1:  # 如果找到了标题字符串
                font_info = fonts[page][start_index:start_index + len(title)]
                # 假设所有字符的字体相同，只取第一个字符的字体信息作为整个字符串的字体信息
                if font_info:  # 确保font_info非空
                    first_char_font = font_info[0][1], font_info[0][2]  # 字体名和大小
                    annotated_titles.append((title, first_char_font))
            else:
                # 如果标题在文本中未找到，可以选择添加一个默认值或留空
                annotated_titles.append((title, ("Not found", 0)))
        annotated_results[page] = annotated_titles
    return annotated_results


In [78]:
#匹配结果
annotated_result = annotate_titles_with_font_info(result_for_match, new_texts, fonts)
print(annotated_result)

{1: [('DeepResidualLearningforImageRecognition', ('XORMUP+NimbusRomNo9L-Medi', 14.346200000000067)), ('KaimingHe', ('CCHXUK+NimbusRomNo9L-Regu', 11.95519999999999)), ('XiangyuZhang', ('CCHXUK+NimbusRomNo9L-Regu', 11.95519999999999)), ('ShaoqingRen', ('CCHXUK+NimbusRomNo9L-Regu', 11.95519999999999)), ('JianSun', ('CCHXUK+NimbusRomNo9L-Regu', 11.95519999999999)), ('MicrosoftResearch', ('CCHXUK+NimbusRomNo9L-Regu', 11.95519999999999)), ('Abstract', ('XORMUP+NimbusRomNo9L-Medi', 11.95519999999999)), ('onCIFAR-10with100and1000layers.', ('RRPAQA+NimbusRomNo9L-ReguItal', 9.962600000000009)), ('ization,COCOdetection,andCOCOsegmentation.', ('RRPAQA+NimbusRomNo9L-ReguItal', 9.962600000000009)), ('1.Introduction', ('XORMUP+NimbusRomNo9L-Medi', 11.95519999999999)), ('1http://image-net.org/challenges/LSVRC/2015/', ('CCHXUK+NimbusRomNo9L-Regu', 5.977599999999995)), ('onImageNetispresentedinFig.4.', ('CCHXUK+NimbusRomNo9L-Regu', 8.966400000000021)), ('greatlybeneﬁtedfromverydeepmodels.', ('CCHXUK+Nim

In [79]:
# 筛选出符合字体大小条件的字符串及其字体信息
toler_rate = 0.001

def extract_strings_by_font_size(annotated_result, main_text_size):
    extracted_strings = {}
    for page, string_tuples in annotated_result.items():
        filtered_tuples = [tuple for tuple in string_tuples if tuple[1][1] > main_text_size * (1+toler_rate)]
        if filtered_tuples:
            extracted_strings[page] = filtered_tuples
    return extracted_strings

In [80]:
main_text_size

9.962600000000009

In [81]:
main_text_size * (1+toler_rate)

9.972562600000009

In [82]:
extracted_strings = extract_strings_by_font_size(annotated_result,main_text_size)
print(extracted_strings)

{1: [('DeepResidualLearningforImageRecognition', ('XORMUP+NimbusRomNo9L-Medi', 14.346200000000067)), ('KaimingHe', ('CCHXUK+NimbusRomNo9L-Regu', 11.95519999999999)), ('XiangyuZhang', ('CCHXUK+NimbusRomNo9L-Regu', 11.95519999999999)), ('ShaoqingRen', ('CCHXUK+NimbusRomNo9L-Regu', 11.95519999999999)), ('JianSun', ('CCHXUK+NimbusRomNo9L-Regu', 11.95519999999999)), ('MicrosoftResearch', ('CCHXUK+NimbusRomNo9L-Regu', 11.95519999999999)), ('Abstract', ('XORMUP+NimbusRomNo9L-Medi', 11.95519999999999)), ('1.Introduction', ('XORMUP+NimbusRomNo9L-Medi', 11.95519999999999))], 2: [('2.RelatedWork', ('XORMUP+NimbusRomNo9L-Medi', 11.95519999999999))], 3: [('3.DeepResidualLearning', ('XORMUP+NimbusRomNo9L-Medi', 11.95519999999999)), ('3.1.ResidualLearning', ('XORMUP+NimbusRomNo9L-Medi', 10.958900000000085)), ('3.2.IdentityMappingbyShortcuts', ('XORMUP+NimbusRomNo9L-Medi', 10.9589)), ('3.3.NetworkArchitectures', ('XORMUP+NimbusRomNo9L-Medi', 10.958899999999971))], 4: [('3.4.Implementation', ('XORMUP+N

In [83]:
#统计extracted_strings中各类字体出现的次数。
def count_font_occurrences(extracted_strings):
    font_counts = {}
    for page, string_tuples in extracted_strings.items():
        for _, font_info in string_tuples:
            font_name = font_info[0]
            if font_name in font_counts:
                font_counts[font_name] += 1
            else:
                font_counts[font_name] = 1
    return font_counts

In [84]:
# 调用函数，统计字体出现次数
font_counts = count_font_occurrences(extracted_strings)
print(font_counts)

{'XORMUP+NimbusRomNo9L-Medi': 17, 'CCHXUK+NimbusRomNo9L-Regu': 5}


In [85]:
def extract_strings_by_most_common_font(extracted_strings):

    font_counts = {}
    for page, string_tuples in extracted_strings.items():
        for string, font_info in string_tuples:
            font_name = font_info[0]
            if font_name not in font_counts:
                font_counts[font_name] = {'count': 0, 'strings': []}
            font_counts[font_name]['count'] += 1
            font_counts[font_name]['strings'].append(string)
    
    # 确定出现次数最多的字体
    most_common_font = max(font_counts.items(), key=lambda x: x[1]['count'])[0]
    
    # 提取出现次数最多的字体对应的字符串
    most_common_font_strings = font_counts[most_common_font]['strings']
    
    return most_common_font_strings

In [86]:
# 调用函数
extracted_titles = extract_strings_by_most_common_font(extracted_strings)
print(f"Strings with the most common font: {extracted_titles}")

Strings with the most common font: ['DeepResidualLearningforImageRecognition', 'Abstract', '1.Introduction', '2.RelatedWork', '3.DeepResidualLearning', '3.1.ResidualLearning', '3.2.IdentityMappingbyShortcuts', '3.3.NetworkArchitectures', '3.4.Implementation', '4.Experiments', '4.1.ImageNetClassiﬁcation', '4.2.CIFAR-10andAnalysis', '4.3.ObjectDetectiononPASCALandMSCOCO', 'References', 'A.ObjectDetectionBaselines', 'B.ObjectDetectionImprovements', 'C.ImageNetLocalization']


In [87]:
# 函数：为列表中的每个标题构建正则表达式，并在给定文本中寻找匹配
def find_titles_with_spaces(titles, full_text):
    located_titles = []
    for title in titles:
        # 为每个字符（包括小数点）之间插入正则表达式，匹配任意数量的空格
        regex_pattern = "\s*".join(re.escape(char) for char in title)
        # 在全文中查找匹配的标题
        match = re.search(regex_pattern, full_text)
        if match:
            located_titles.append(match.group().strip())
    return located_titles

In [88]:
# 使用函数匹配含有空格的标题
located_titles = find_titles_with_spaces(extracted_titles, full_pdf_text)
print(located_titles)

['Deep Residual Learning for Image Recognition', 'Abstract', '1. Introduction', '2. Related Work', '3. Deep Residual Learning', '3.1. Residual Learning', '3.2. Identity Mapping by Shortcuts', '3.3. Network Architectures', '3.4. Implementation', '4. Experiments', '4.1. ImageNet Classiﬁcation', '4.2. CIFAR-10 and Analysis', '4.3. Object Detection on PASCAL and MS COCO', 'References', 'A. Object Detection Baselines', 'B. Object Detection Improvements', 'C. ImageNet Localization']


In [89]:
#截取标题与标题之间的内容作为前一个标题下的所属内容
def extract_content_between_titles(located_titles, full_text):
    structurized_content = {}
    # 构建每个标题的正则表达式模式，并确保按照标题在原文中的出现顺序进行匹配
    title_patterns = [re.escape(title) for title in located_titles]
    title_regex = '(' + '|'.join(title_patterns) + ')'
    # 使用正则表达式分割全文，保留标题作为分隔符
    parts = re.split(title_regex, full_text)
    # 初始化变量用于存储当前处理的标题
    current_title = None
    for part in parts:
        if part in located_titles:
            # 遇到新标题，更新当前标题，并初始化对应的内容为空字符串
            current_title = part
            structurized_content[current_title] = ''
        elif current_title:
            # 非标题部分，追加到当前标题对应的内容中
            structurized_content[current_title] += part

    return structurized_content

In [90]:
# 提取内容
structurized_content_detail = extract_content_between_titles(located_titles, full_pdf_text)

In [91]:
# 打印结果
for title, content in structurized_content_detail.items():
    print(f'Title: {title}\nContent: {content}\n')

Title: Deep Residual Learning for Image Recognition
Content: 

Kaiming He

Xiangyu Zhang

Shaoqing Ren

Jian Sun

Microsoft Research

@microsoft.com
kahe, v-xiangz, v-shren, jiansun
}
{

5
1
0
2
c
e
D
0
1

]

V
C
.
s
c
[

1
v
5
8
3
3
0
.
2
1
5
1
:
v
i
X
r
a



Title: Abstract
Content: 

Deeper neural networks are more difﬁcult to train. We
present a residual learning framework to ease the training
of networks that are substantially deeper than those used
previously. We explicitly reformulate the layers as learn-
ing residual functions with reference to the layer inputs, in-
stead of learning unreferenced functions. We provide com-
prehensive empirical evidence showing that these residual
networks are easier to optimize, and can gain accuracy from
considerably increased depth. On the ImageNet dataset we
evaluate residual nets with a depth of up to 152 layers—8
×
deeper than VGG nets [41] but still having lower complex-
ity. An ensemble of these residual nets achieves 3.57% error
on the 

In [92]:
for title in structurized_content_detail.keys():
    print(title)

Deep Residual Learning for Image Recognition
Abstract
1. Introduction
2. Related Work
3. Deep Residual Learning
3.1. Residual Learning
3.2. Identity Mapping by Shortcuts
3.3. Network Architectures
3.4. Implementation
4. Experiments
4.1. ImageNet Classiﬁcation
4.2. CIFAR-10 and Analysis
4.3. Object Detection on PASCAL and MS COCO
References
A. Object Detection Baselines
B. Object Detection Improvements
C. ImageNet Localization


In [93]:
#合并一级标题下的内容，只按一级标题的层次来划分
def merge_contents_under_primary_titles(titles_content):
    structurized_full_text = {}
    current_primary_title = None

    for title, content in titles_content.items():
        # 调整正则表达式以匹配数字开头，可能紧跟点或空格的一级标题
        if re.match(r'^[A-Z]|^\d+\s*(\.?\s)', title):
            current_primary_title = title
            structurized_full_text[current_primary_title] = content
        else:
            if current_primary_title:
                # 将二级标题及其下的内容追加到当前一级标题下
                # 这里包括了二级标题本身作为内容的一部分
                structurized_full_text[current_primary_title] += " " + title + " " + content

    return structurized_full_text

In [94]:
# 调用函数
structurized_full_text = merge_contents_under_primary_titles(structurized_content_detail)

In [95]:
# 打印结果
for title, content in structurized_full_text.items():
    print(f'Title: {title}\nContent: {content}\n')

Title: Deep Residual Learning for Image Recognition
Content: 

Kaiming He

Xiangyu Zhang

Shaoqing Ren

Jian Sun

Microsoft Research

@microsoft.com
kahe, v-xiangz, v-shren, jiansun
}
{

5
1
0
2
c
e
D
0
1

]

V
C
.
s
c
[

1
v
5
8
3
3
0
.
2
1
5
1
:
v
i
X
r
a



Title: Abstract
Content: 

Deeper neural networks are more difﬁcult to train. We
present a residual learning framework to ease the training
of networks that are substantially deeper than those used
previously. We explicitly reformulate the layers as learn-
ing residual functions with reference to the layer inputs, in-
stead of learning unreferenced functions. We provide com-
prehensive empirical evidence showing that these residual
networks are easier to optimize, and can gain accuracy from
considerably increased depth. On the ImageNet dataset we
evaluate residual nets with a depth of up to 152 layers—8
×
deeper than VGG nets [41] but still having lower complex-
ity. An ensemble of these residual nets achieves 3.57% error
on the 

In [96]:
# 打印structurized_full_text字典的所有键
for title in structurized_full_text.keys():
    print(title)


Deep Residual Learning for Image Recognition
Abstract
1. Introduction
2. Related Work
3. Deep Residual Learning
4. Experiments
References
A. Object Detection Baselines
B. Object Detection Improvements
C. ImageNet Localization
